# 🗡️ YOLOv26-Based Kris Detection and Classification for Madura Cultural Heritage Preservation
## Final Notebook Uji Coba Model - Google Colab T4 GPU

Notebook ini dibuat untuk kebutuhan penelitian tesis **"YOLOv26-Based Kris Detection and Classification for Madura Cultural Heritage Preservation"**.

### Alur Eksperimen:
1. **Setup & Instalasi** - Pemasangan pustaka pendukung (Ultralytics, OpenCV, GDM, dll.).
2. **Knowledge Base Budaya** - Integrasi data Dapur, Pamor, Tangguh, dan Luk.
3. **Dataset Crawling & Parsing** - Skrip download otomatis gambar & metadata kris.
4. **Training Model YOLOv26** - Konfigurasi data YAML dan fine-tuning model.
5. **Inferensi & Visualisasi** - Deteksi bilah keris & visualisasi ringkasan metadata budaya.

## 1. 🔧 Setup & GPU Check

In [ ]:
# Cek ketersediaan GPU NVIDIA di Google Colab
!nvidia-smi

# Pasang dependencies utama
!pip install -q 'ultralytics>=8.3.50' beautifulsoup4 pillow requests pandas tqdm opencv-python matplotlib

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO
import requests
import cv2
import os
import json
import re
import pandas as pd
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
print(f"Ultralytics version: {ultralytics.__version__}")

## 2. 📚 Madura Cultural Kris Knowledge Base

In [ ]:
KERIS_KNOWLEDGE_BASE = {
    'dapur': {
        'brojol': {
            'nama': 'Brojol',
            'luk': 0,
            'deskripsi': 'Bilah lurus tanpa luk, melambangkan kesederhanaan jiwa.',
            'filosofi': 'Kepolosan dan kemurnian spiritual.'
        },
        'tilam_upih': {
            'nama': 'Tilam Upih',
            'luk': 0,
            'deskripsi': 'Bilah lurus, simbol ketenangan rumah tangga.',
            'filosofi': 'Keharmonisan dan perlindungan keluarga.'
        },
        'sengkelat': {
            'nama': 'Sengkelat',
            'luk': 13,
            'deskripsi': 'Bilah luk 13, sakral dan melambangkan perjuangan rakyat.',
            'filosofi': 'Keberanian, kewibawaan spiritual tertinggi.'
        },
        'jalak': {
            'nama': 'Jalak',
            'luk': 0,
            'deskripsi': 'Bilah lurus dengan gandik burung jalak.',
            'filosofi': 'Kebebasan jiwa, kegunaan sosial yang luas.'
        }
    },
    'pamor': {
        'beras_wutah': {
            'nama': 'Beras Wutah',
            'makna': 'Rezeki yang melimpah dan kemakmuran keluarga.',
            'proses': 'Lipatan mlumah nikel meteorit.'
        },
        'ngulit_semangka': {
            'nama': 'Ngulit Semangka',
            'makna': 'Mempermudah pergaulan, memperbanyak rezeki.',
            'proses': 'Pamor miring silang nikel.'
        }
    },
    'tangguh': {
        'madura': {
            'nama': 'Tangguh Madura',
            'periode': 'Abad ke-17 - Sekarang',
            'ciri': 'Bilah tebal, baja gelap padat, ganja pipih lebar.'
        }
    }
}

## 3. 🕸️ Dataset Preparation: Crawler Module

In [ ]:
# Konfigurasi Crawler
BASE_URL = 'https://pusakakeris.com'
KATALOG_URL = 'https://pusakakeris.com/katalog/'
OUTPUT_DIR = './dataset_keris_colab'
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'
}

def get_sample_urls(limit_pages=3):
    urls = []
    print(f"Scanning {limit_pages} halaman katalog...")
    for p in range(1, limit_pages + 1):
        page_url = KATALOG_URL if p == 1 else f"{KATALOG_URL}page/{p}/"
        try:
            r = requests.get(page_url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(r.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a['href']
                if 'pusakakeris.com/keris-' in href and href not in urls:
                    urls.append(href)
        except Exception as e:
            print(f"Skip page {p}: {e}")
    return urls

urls = get_sample_urls(3)
print(f"Ditemukan {len(urls)} produk keris. Siap diunduh.")

## 4. 🏋️ YOLOv26 Model Configuration & Fine-Tuning

In [ ]:
# Muat model YOLO26n pretrained untuk objek deteksi
model = YOLO('yolo26n.pt')
print("Model Parameters:", sum(p.numel() for p in model.model.parameters()))

In [ ]:
# Contoh konfigurasi dataset YAML untuk YOLO training
dataset_yaml = """
path: ./dataset_keris_colab
train: images/train
val: images/val

names:
  0: keris_lurus
  1: keris_luk_3
  2: keris_luk_5
  3: keris_luk_7
  4: keris_luk_9
  5: keris_luk_11
  6: keris_luk_13
  7: keris_madura
"""

with open("keris_dataset.yaml", "w") as f:
    f.write(dataset_yaml.strip())
print("File keris_dataset.yaml berhasil ditulis.")

In [ ]:
# Jalankan training model YOLOv26 (Ganti epoch & path sesuai kebutuhan tesis)
# model.train(data='keris_dataset.yaml', epochs=30, imgsz=640, device=0)

## 5. 🔍 Inferensi & Linking ke Knowledge Base Budaya

In [ ]:
def show_cultural_details(class_name, conf):
    kb = KERIS_KNOWLEDGE_BASE
    
    # Fallback/matching
    d_info = kb['dapur']['brojol']
    if 'sengkelat' in class_name:
        d_info = kb['dapur']['sengkelat']
    elif 'jalak' in class_name:
        d_info = kb['dapur']['jalak']
        
    p_info = kb['pamor']['beras_wutah']
    t_info = kb['tangguh']['madura']
    
    print("="*50)
    print(f"🎯 TERDETEKSI: {class_name.upper()} (Akurasi: {conf:.2%})")
    print(f"⚔️ Dapur      : {d_info['nama']} - {d_info['deskripsi']}")
    print(f"🌀 Filosofi   : {d_info['filosofi']}")
    print(f"✨ Pamor      : {p_info['nama']} ({p_info['makna']})")
    print(f"📅 Tangguh    : {t_info['nama']} ({t_info['periode']})")
    print("="*50)

# Uji coba fungsi lookup
show_cultural_details("keris_sengkelat", 0.945)